# grid_py Tutorial 2: WebRenderer Interactive Output

This notebook demonstrates the **WebRenderer** backend, which produces interactive HTML with:
- SVG rendering for structural elements (text, axes, few-element geoms)
- Canvas rendering for data-dense elements (>2000 points)
- D3.js-powered zoom, pan, hover tooltip, and brush selection

The API is **identical** to CairoRenderer — just swap the renderer.

In [ ]:
import numpy as np
from IPython.display import display, HTML

from grid_py import (
    WebRenderer, Gpar, Unit, Viewport, GridLayout,
    get_state, grid_draw,
    push_viewport, pop_viewport,
    rect_grob, circle_grob, text_grob, lines_grob,
    points_grob, polygon_grob, segments_grob, roundrect_grob,
)

def show_web(renderer):
    """Display WebRenderer output inline in Jupyter."""
    display(HTML(renderer.to_html()))

## 1. Simple Interactive Plot

Hover over elements to see tooltips. Try zooming with the scroll wheel and panning by dragging.

In [ ]:
r = WebRenderer(width=6, height=4, dpi=100)
state = get_state()
state.init_device(r)

# Background
grid_draw(rect_grob(x=0.5, y=0.5, width=1.0, height=1.0,
                     gp=Gpar(fill='#f8f9fa', col='#dee2e6')))

# Some colorful shapes
colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#264653']
for i, (cx, cy, color) in enumerate([
    (0.2, 0.3, colors[0]),
    (0.4, 0.7, colors[1]),
    (0.6, 0.4, colors[2]),
    (0.8, 0.6, colors[3]),
    (0.5, 0.5, colors[4]),
]):
    grid_draw(circle_grob(x=cx, y=cy, r=0.08,
                           gp=Gpar(fill=color, col='white', lwd=2)))

grid_draw(text_grob(label='Interactive: zoom & pan with mouse',
                     x=0.5, y=0.95,
                     gp=Gpar(fontsize=13, fontface='bold', col='#264653')))

show_web(r)

## 2. Scatter Plot with Tooltip Data

When you hover over points, the tooltip shows the attached metadata (species, petal length).

This uses `grob.metadata` to attach data that flows through to the scene graph.

In [ ]:
np.random.seed(42)

# Simulated iris-like data
n_per = 25
species = ['setosa'] * n_per + ['versicolor'] * n_per + ['virginica'] * n_per
petal_length = np.concatenate([
    np.random.normal(1.5, 0.3, n_per),
    np.random.normal(4.3, 0.5, n_per),
    np.random.normal(5.5, 0.5, n_per),
])
petal_width = np.concatenate([
    np.random.normal(0.3, 0.1, n_per),
    np.random.normal(1.3, 0.2, n_per),
    np.random.normal(2.0, 0.3, n_per),
])

# Normalize to NPC
x_npc = 0.1 + 0.8 * (petal_length - petal_length.min()) / (petal_length.max() - petal_length.min())
y_npc = 0.12 + 0.75 * (petal_width - petal_width.min()) / (petal_width.max() - petal_width.min())

# Colors per species
color_map = {'setosa': '#e63946', 'versicolor': '#457b9d', 'virginica': '#2a9d8f'}
cols = [color_map[s] for s in species]

r = WebRenderer(width=7, height=5, dpi=100)
state.init_device(r)

# Plot area
grid_draw(rect_grob(x=0.5, y=0.5, width=1.0, height=1.0,
                     gp=Gpar(fill='white', col='#ccc')))

# Points with metadata for tooltips
grob = points_grob(x=x_npc.tolist(), y=y_npc.tolist(), pch=19,
                    gp=Gpar(col=cols, fill=cols, fontsize=6, alpha=0.8))
grob.metadata = {
    'species': species,
    'petal_length': [f'{v:.1f}' for v in petal_length],
    'petal_width': [f'{v:.1f}' for v in petal_width],
}
grid_draw(grob)

# Axis labels
grid_draw(text_grob(label='Petal Length', x=0.5, y=0.03,
                     gp=Gpar(fontsize=11, col='#333')))
grid_draw(text_grob(label='Petal Width', x=0.03, y=0.5, rot=90,
                     gp=Gpar(fontsize=11, col='#333')))

# Title
grid_draw(text_grob(label='Iris Scatter (hover for tooltip)', x=0.5, y=0.96,
                     gp=Gpar(fontsize=14, fontface='bold', col='#264653')))

# Legend
for i, (sp, color) in enumerate(color_map.items()):
    ypos = 0.88 - i * 0.06
    grid_draw(circle_grob(x=0.82, y=ypos, r=0.01,
                           gp=Gpar(fill=color, col=color)))
    grid_draw(text_grob(label=sp, x=0.85, y=ypos, hjust=0,
                         gp=Gpar(fontsize=9, col='#333')))

show_web(r)

## 3. Viewport Layout (same as Cairo, but interactive)

In [ ]:
r = WebRenderer(width=7, height=5, dpi=100, theme='light')
state.init_device(r)

# Title
grid_draw(text_grob(label='Multi-Panel Layout', x=0.5, y=0.97,
                     gp=Gpar(fontsize=15, fontface='bold', col='#2c3e50')))

# 1x3 layout
layout = GridLayout(nrow=1, ncol=3,
                     widths=Unit([1, 1, 1], 'null'),
                     heights=Unit([1], 'null'))

main_vp = Viewport(name='main', x=0.5, y=0.47, width=0.92, height=0.85, layout=layout)
push_viewport(main_vp, recording=False)
r.push_viewport(main_vp)

panel_titles = ['Panel A', 'Panel B', 'Panel C']
panel_colors = ['#e63946', '#457b9d', '#2a9d8f']

for col in [1, 2, 3]:
    vp = Viewport(name=f'panel_{col}', layout_pos_row=1, layout_pos_col=col)
    push_viewport(vp, recording=False)
    r.push_viewport(vp)

    # Panel background
    grid_draw(rect_grob(x=0.5, y=0.5, width=0.92, height=0.92,
                         gp=Gpar(fill='#f8f9fa', col=panel_colors[col-1], lwd=2)))

    # Random scatter in each panel
    np.random.seed(col)
    px = np.random.rand(20) * 0.7 + 0.15
    py = np.random.rand(20) * 0.6 + 0.15
    grid_draw(points_grob(x=px.tolist(), y=py.tolist(), pch=19,
                           gp=Gpar(col=panel_colors[col-1],
                                   fill=panel_colors[col-1], fontsize=5)))

    # Panel title
    grid_draw(text_grob(label=panel_titles[col-1], x=0.5, y=0.92,
                         gp=Gpar(fontsize=11, fontface='bold',
                                 col=panel_colors[col-1])))

    pop_viewport(1, recording=False)
    r.pop_viewport()

pop_viewport(1, recording=False)
r.pop_viewport()

show_web(r)

## 4. Inspect the Scene Graph

The WebRenderer builds a JSON scene graph internally. You can inspect it directly.

In [ ]:
import json

# Print the scene graph tree structure
sg = json.loads(r.to_scene_json())

def print_tree(node, indent=0):
    t = node.get('type', '?')
    name = node.get('name', '')
    hint = node.get('render_hint', '')
    extra = f' name="{name}"' if name else ''
    extra += f' hint={hint}' if hint else ''
    print(' ' * indent + f'{t}{extra}')
    for child in node.get('children', []):
        print_tree(child, indent + 2)

print(f'Scene graph: {sg["width"]}x{sg["height"]}px, {sg["dpi"]} dpi')
print(f'Defs: {len(sg["defs"]["gradients"])} gradients, '
      f'{len(sg["defs"]["clip_paths"])} clips, '
      f'{len(sg["defs"]["masks"])} masks')
print()
print_tree(sg['root'])

## 5. Save as Standalone HTML

The HTML file is self-contained — CSS, JavaScript runtime, D3.js CDN link, and scene graph data are all embedded.

In [ ]:
r.save('interactive_plot.html')
print('Saved interactive_plot.html')
print(f'File size: {len(r.to_html()) / 1024:.1f} KB')
print('Open in browser for full interactivity (zoom, pan, tooltips).')